# 07 Feature Fusion — SENTINEL
Output for 08: `artifacts/X_train.npz`, `X_valid.npz`, `X_test.npz`, `y_*.npy`, `feature_names.json`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))


In [ ]:
import json, joblib, numpy as np, pandas as pd
from scipy.sparse import save_npz, csr_matrix
from src.feature_fusion import fuse_features
from src import config


In [ ]:
train = pd.read_csv(config.ARTIFACTS_DIR / 'train_fe.csv')
valid = pd.read_csv(config.ARTIFACTS_DIR / 'valid_fe.csv')
test = pd.read_csv(config.ARTIFACTS_DIR / 'test_fe.csv')
emb = joblib.load(config.ARTIFACTS_DIR / 'graph_emb.joblib')
Xtr, names = fuse_features(train, emb)
Xva, _ = fuse_features(valid, emb)
Xte, _ = fuse_features(test, emb)
print(Xtr.shape, Xva.shape, Xte.shape, len(names))


In [ ]:
print('n_feats', len(names))
print(names[:10])
print(pd.DataFrame(Xtr.values, columns=names).describe().T.head(10))


In [ ]:
save_npz(config.ARTIFACTS_DIR / 'X_train.npz', csr_matrix(Xtr.values))
save_npz(config.ARTIFACTS_DIR / 'X_valid.npz', csr_matrix(Xva.values))
save_npz(config.ARTIFACTS_DIR / 'X_test.npz', csr_matrix(Xte.values))
np.save(config.ARTIFACTS_DIR / 'y_train.npy', train['is_suspicious'].values)
np.save(config.ARTIFACTS_DIR / 'y_valid.npy', valid['is_suspicious'].values)
np.save(config.ARTIFACTS_DIR / 'y_test.npy', test['is_suspicious'].values)
(config.ARTIFACTS_DIR / 'feature_names.json').write_text(json.dumps(names, indent=2))
print('saved X_*.npz + y_*.npy + feature_names.json -> used by 08_xgboost_training.ipynb')
